# 01 — EDA: Exploratory Data Analysis
**Goal:** Understand all 4 datasets, spot patterns, and validate the data before building any models.

Datasets:
- `equity_dataset.csv` — Equity price, returns, SMA  
- `macro_dataset.csv` — Inflation, Interest rates, USD, Sentiment  
- `multi_asset_dataset.csv` — Oil, Gold, Bonds  
- `oil_dataset.csv` — Oil price + volatility

In [ ]:
import sys
sys.path.append('../src')  # so we can import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from ingestion import load_master, validate

plt.style.use('dark_background')
sns.set_palette('husl')

# Load everything in one shot
df = load_master()
validate(df)
df.head()

## 1. Dataset Overview

In [ ]:
print(f"Total rows       : {len(df):,}")
print(f"Date range       : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Trading days/yr  : {len(df) / df['Date'].dt.year.nunique():.0f}")
print(f"\nAll columns:")
for col in df.columns:
    print(f"  {col:<25} | NaN: {df[col].isna().sum():>5} | dtype: {df[col].dtype}")

## 2. Equity Price Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Price
axes[0].plot(df['Date'], df['Equity_Price'], color='#00d4ff', lw=1, label='Price')
axes[0].plot(df['Date'], df['Equity_SMA10'], color='#ff6b35', lw=1, alpha=0.8, label='SMA10')
axes[0].set_title('Equity Price & 10-Day SMA', fontsize=14)
axes[0].set_ylabel('Price ($)')
axes[0].legend()

# Volume
axes[1].bar(df['Date'], df['Equity_Volume'], color='#7c3aed', alpha=0.6, width=1)
axes[1].set_title('Daily Volume', fontsize=14)
axes[1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

## 3. Returns Distribution

In [ ]:
returns = df['Equity_Returns'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(returns, bins=100, color='#00d4ff', alpha=0.8, edgecolor='none')
axes[0].axvline(returns.mean(), color='#ff6b35', lw=2, label=f'Mean: {returns.mean():.4f}')
axes[0].axvline(returns.quantile(0.05), color='red', lw=1.5, linestyle='--', label=f'5% VaR: {returns.quantile(0.05):.4f}')
axes[0].set_title('Return Distribution')
axes[0].legend(fontsize=9)

# Rolling returns over time
axes[1].plot(df['Date'], returns, color='#00d4ff', alpha=0.5, lw=0.5)
axes[1].axhline(0, color='white', lw=0.5, linestyle='--')
axes[1].set_title('Daily Returns Over Time')

# Cumulative return
cumret = (1 + returns).cumprod()
axes[2].plot(df['Date'].iloc[1:], cumret.values, color='#10b981', lw=1.5)
axes[2].set_title('Cumulative Return')
axes[2].set_ylabel('Portfolio Value (starting at 1)')

plt.tight_layout()
plt.show()

print(f"Annualized Return : {returns.mean() * 365:.2%}")
print(f"Annualized Std    : {returns.std() * np.sqrt(365):.2%}")
print(f"Sharpe (0% RFR)   : {(returns.mean() / returns.std()) * np.sqrt(365):.3f}")
print(f"Max Daily Return  : {returns.max():.4f}")
print(f"Min Daily Return  : {returns.min():.4f}")
print(f"Skewness          : {returns.skew():.4f}")
print(f"Kurtosis          : {returns.kurtosis():.4f}")

## 4. Multi-Asset Comparison

In [ ]:
# Normalize prices to 100 at start for fair comparison
assets = {
    'Equity' : 'Equity_Price',
    'Oil'    : 'Oil_Price',
    'Gold'   : 'MA_Gold_Price',
    'Bonds'  : 'MA_Bonds_Price'
}

colors = ['#00d4ff', '#f59e0b', '#fbbf24', '#10b981']

fig, ax = plt.subplots(figsize=(16, 6))
for (name, col), color in zip(assets.items(), colors):
    series = df[col].dropna()
    normalized = (series / series.iloc[0]) * 100
    ax.plot(df['Date'].iloc[:len(series)], normalized, label=name, lw=1.2, color=color)

ax.axhline(100, color='white', lw=0.5, linestyle='--', alpha=0.5)
ax.set_title('Asset Prices Normalized to 100 (Base = Jan 2020)', fontsize=14)
ax.set_ylabel('Normalized Price')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
return_cols = [
    'Equity_Returns', 'Oil_Returns', 'MA_Oil_Returns',
    'MA_Gold_Returns', 'Inflation', 'Interest_Rate',
    'USD_Index', 'Sentiment'
]

corr = df[return_cols].dropna().corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True
)
plt.title('Correlation Matrix — Returns & Macro Signals', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Macro Signals Over Time

In [ ]:
macro_cols = ['Inflation', 'Interest_Rate', 'USD_Index', 'Sentiment']
colors_macro = ['#f87171', '#fb923c', '#a78bfa', '#34d399']

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
for ax, col, color in zip(axes, macro_cols, colors_macro):
    ax.plot(df['Date'], df[col], color=color, lw=0.8)
    # Rolling 30-day average
    ax.plot(df['Date'], df[col].rolling(30).mean(), color='white', lw=1.5, alpha=0.7, linestyle='--')
    ax.set_ylabel(col, fontsize=10)
    ax.set_title(col, fontsize=11)

plt.suptitle('Macro Signals Over Time (dashed = 30-day avg)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Oil Volatility

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].plot(df['Date'], df['Oil_Price'], color='#f59e0b', lw=1)
axes[0].set_title('Oil Price')

axes[1].fill_between(df['Date'], df['Oil_Volatility'], color='#f87171', alpha=0.7)
axes[1].set_title('Oil Volatility (Rolling Std)')

plt.tight_layout()
plt.show()

print(f"Mean Oil Volatility : {df['Oil_Volatility'].mean():.5f}")
print(f"Peak Oil Volatility : {df['Oil_Volatility'].max():.5f} on {df.loc[df['Oil_Volatility'].idxmax(), 'Date'].date()}")

## 8. Next Steps

From this EDA, you should now know:
- ✅ What the overall price trend looks like
- ✅ Which assets are correlated
- ✅ How volatile the data is
- ✅ What macro regimes exist

**→ Go to `02_preprocessing.ipynb` to clean the data and engineer features.**